# Term Structure (Calendar Spread) Strategy

This notebook demonstrates the `TermStructureStrategy` which trades mean-reversion between near-term and long-term implied volatility. When the spread (far - near) widens beyond a threshold relative to its rolling z-score, the strategy takes a spread trade expecting convergence.

In [ ]:
from __future__ import annotations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from backtest.engine import BacktestEngine
from strategies.term_structure import TermStructureStrategy

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

## Generate Synthetic IV Term Structure Data

We'll create synthetic data with near-term and far-term implied volatility that exhibits mean-reverting spread behavior.

In [ ]:
def make_term_structure_data(n: int = 500, seed: int = 42) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    dates = pd.date_range("2024-01-01", periods=n, freq="1h")
    # Simulate underlying price
    r = rng.normal(0.0, 0.01, n)
    price = 100 * np.exp(np.cumsum(r))
    
    # Near-term IV oscillates faster with larger swings
    t = np.arange(n, dtype=float)
    iv_30d = 0.3 + 0.08 * np.sin(t / 3.0) + rng.normal(0, 0.015, n)
    # Far-term IV is smoother, lower mean reversion speed
    iv_90d = 0.32 + 0.04 * np.sin(t / 8.0) + rng.normal(0, 0.008, n)
    iv_30d = np.clip(iv_30d, 0.15, 0.6)
    iv_90d = np.clip(iv_90d, 0.15, 0.6)
    
    return pd.DataFrame(
        {
            "BTC": price,
            "iv_30d": iv_30d,
            "iv_90d": iv_90d,
        },
        index=dates,
    )

data = make_term_structure_data()
print(f"Data shape: {data.shape}")
data.head()

## Instantiate and Run the Strategy

In [ ]:
# Create strategy instance
strat = TermStructureStrategy(
    asset="BTC",
    near_tenor="iv_30d",
    far_tenor="iv_90d",
    lookback=30,
    entry_threshold=2.0,
    exit_threshold=0.5,
)

# Run backtest
engine = BacktestEngine(initial_capital=100_000.0)
result = engine.run(strat, data)

# Also run with benchmark for comparison
result_bench = engine.run_with_benchmark(strat, data)

## Display Performance Metrics

In [ ]:
from utils.reporting import generate_report

print("=== Strategy Performance ===")
print(generate_report(result))
print("\n=== With Benchmark ===")
print(f"Strategy Return: {result_bench['total_return']:.4f}")
print(f"Benchmark Return: {result_bench['benchmark_return']:.4f}")
print(f"Excess Return: {result_bench['excess_return']:.4f}")

## Visualize Spread, Z-Score, and Trading Signals

In [ ]:
signals = strat.generate_signals(data)

fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True)

# Plot 1: IV Term Structure
axes[0].plot(data.index, data['iv_30d'], label='IV 30d (Near)', alpha=0.7)
axes[0].plot(data.index, data['iv_90d'], label='IV 90d (Far)', alpha=0.7)
axes[0].set_ylabel('Implied Volatility')
axes[0].legend()
axes[0].set_title('Volatility Term Structure')

# Plot 2: Spread (Far - Near)
axes[1].plot(signals.index, signals['spread'], label='Spread (Far - Near)', color='purple')
axes[1].axhline(y=0, color='black', linestyle='--', alpha=0.5)
axes[1].set_ylabel('Spread')
axes[1].legend()
axes[1].set_title('IV Spread')

# Plot 3: Z-Score with thresholds
axes[2].plot(signals.index, signals['spread_zscore'], label='Spread Z-Score', color='orange')
axes[2].axhline(y=strat.entry_threshold, color='red', linestyle='--', label=f'Entry (+{strat.entry_threshold})')
axes[2].axhline(y=-strat.entry_threshold, color='green', linestyle='--', label=f'Entry (-{strat.entry_threshold})')
axes[2].axhline(y=strat.exit_threshold, color='gray', linestyle=':', alpha=0.7)
axes[2].axhline(y=-strat.exit_threshold, color='gray', linestyle=':', alpha=0.7)
axes[2].axhline(y=0, color='black', linestyle='-', alpha=0.3)
axes[2].set_ylabel('Z-Score')
axes[2].legend()
axes[2].set_title('Spread Z-Score and Trading Thresholds')

# Plot 4: Positions/Weights
axes[3].plot(signals.index, signals['BTC'], label='Position Weight', color='blue', drawstyle='steps-post')
axes[3].axhline(y=0, color='black', linestyle='-', alpha=0.3)
axes[3].set_ylabel('Weight')
axes[3].set_xlabel('Date')
axes[3].legend()
axes[3].set_title('Strategy Positions (Long/Short/Flat)')

plt.tight_layout()
plt.show()

## Equity Curve vs Benchmark

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

# Plot equity curves
if 'equity_curve' in result:
    equity = result['equity_curve']
    ax.plot(equity.index, equity.values, label='Term Structure Strategy', linewidth=2)

if 'benchmark_equity' in result_bench:
    bench_equity = result_bench['benchmark_equity']
    ax.plot(bench_equity.index, bench_equity.values, label='Buy & Hold Benchmark', linewidth=2, alpha=0.7)

ax.set_title('Equity Curve: Term Structure Strategy vs Benchmark')
ax.set_xlabel('Date')
ax.set_ylabel('Portfolio Value ($)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Summary Statistics

In [ ]:
# Calculate additional metrics
if 'equity_curve' in result:
    equity = result['equity_curve']
    returns = equity.pct_change().dropna()
    
    print(f"Total Return: {result.get('total_return', 0):.2%}")
    print(f"Sharpe Ratio: {result.get('sharpe_ratio', 0):.4f}")
    print(f"Max Drawdown: {result.get('max_drawdown', 0):.2%}")
    print(f"Win Rate: {result.get('win_rate', 0):.2%}")
    print(f"Number of Trades: {result.get('num_trades', 0)}")
    print(f"Annualized Return: {(1 + result.get('total_return', 0))**(252*24/len(equity)) - 1:.2%}" if len(equity) > 0 else "N/A")
    print(f"Volatility (annualized): {returns.std() * np.sqrt(252*24):.2%}")